# Discursos: classificar por tema + medir o *domain shift* (Fase 6 — parte 1)

Aqui aplicamos o **BERTimbau treinado nas ementas** aos **discursos** dos deputados. Como o
discurso é longo e de outro estilo (oratória), esperamos uma **queda** de desempenho
(*domain shift*) — e vamos **medi-la** usando uma amostra rotulada por uma LLM como gabarito.

**O que este notebook faz:**
1. Recarrega o `modelo_bertimbau/` (não treina) e classifica **todos** os discursos por
   *chunking* (quebra o texto longo em janelas de 192 tokens).
2. Monta uma amostra (~120) e a rotula com uma **LLM** (rota manual: colar no Claude/ChatGPT).
3. Mede o **domain shift por tema** (BERTimbau × gabarito) → diz em quais temas confiar no
   cruzamento (notebook 06).

> Pré-requisitos: `discursos_todos.csv`, `modelo_bertimbau/`, `proposicoes_temas.csv`,
> `resultados_bertimbau.csv`. Instale: `pip install torch transformers scikit-learn pandas`.

## Passo 1 — Carregar discursos, modelo e limiares
Reconstruímos a **ordem dos 32 temas** do mesmo jeito do treino (`MultiLabelBinarizer` sobre
`proposicoes_temas.csv`) — é essa ordem que casa com as 32 saídas do modelo. Pegamos também o
**limiar por tema** salvo em `resultados_bertimbau.csv`.

In [ ]:
import pandas as pd, numpy as np, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.preprocessing import MultiLabelBinarizer

disc = pd.read_csv("discursos_todos.csv", sep=";", dtype=str)
disc["transcricao"] = disc["transcricao"].fillna("").astype(str)
print(f"{len(disc)} discursos | {disc['id_deputado'].nunique()} deputados")

# ordem dos temas = a mesma do treino (classes_ do MLB, alfabetica)
props = pd.read_csv("proposicoes_temas.csv", dtype=str)
mlb = MultiLabelBinarizer().fit(props["temas"].str.split("|"))
nomes_temas = list(mlb.classes_)
assert len(nomes_temas) == 32, nomes_temas

# limiar por tema (na ordem do modelo)
resb = pd.read_csv("resultados_bertimbau.csv")
lim_map = dict(zip(resb["tema"], resb["limiar"]))
limiares = np.array([float(lim_map.get(t, 0.5)) for t in nomes_temas])

tokenizer = AutoTokenizer.from_pretrained("modelo_bertimbau")
modelo = AutoModelForSequenceClassification.from_pretrained("modelo_bertimbau")
device = "cuda" if torch.cuda.is_available() else "cpu"
modelo.to(device).eval()
print("modelo recarregado em", device)

## Passo 2 — Classificar cada discurso por *chunking*
O BERT só lê 192 tokens; o discurso é bem maior. Usamos o próprio tokenizer para fatiar o
texto em **janelas** (`return_overflowing_tokens`), prevemos cada janela e **agregamos por
MAX** (se o tema aparece forte em qualquer trecho, conta para o discurso). Aplicamos os
limiares por tema → multirrótulo previsto.

> É um lote grande (~46 mil discursos). Roda em GPU; estimativa de alguns minutos a ~1h.

In [ ]:
def prob_discurso(texto):
    """Probabilidade (sigmoide) por tema para UM discurso, agregando as janelas por MAX."""
    enc = tokenizer(texto, max_length=192, truncation=True,
                    return_overflowing_tokens=True, padding="max_length",
                    return_tensors="pt")
    with torch.no_grad():
        logits = modelo(input_ids=enc["input_ids"].to(device),
                        attention_mask=enc["attention_mask"].to(device)).logits
    probs = torch.sigmoid(logits).cpu().numpy()   # (n_janelas, 32)
    return probs.max(axis=0)

P = np.zeros((len(disc), 32), dtype="float32")
for i, texto in enumerate(disc["transcricao"].values):
    P[i] = prob_discurso(texto)
    if (i + 1) % 2000 == 0:
        print(f"  ... {i+1}/{len(disc)} discursos classificados")

pred = (P >= limiares).astype(int)
print("Classificacao concluida.")

In [ ]:
# salva os temas previstos por discurso (texto "A|B")
def temas_de(linha):
    ts = [nomes_temas[j] for j in np.where(linha == 1)[0]]
    return "|".join(ts)

disc_out = disc[["id_deputado", "nome_deputado", "data"]].copy()
disc_out["temas_previstos"] = [temas_de(r) for r in pred]
disc_out.to_csv("discursos_classificados.csv", sep=";", index=False, encoding="utf-8")
print("Salvo: discursos_classificados.csv")

# panorama agregado (plausibilidade)
cont = pred.sum(axis=0)
print("\nTemas mais previstos nos discursos:")
for j in np.argsort(-cont)[:10]:
    print(f"  {nomes_temas[j]:<42} {int(cont[j])}")

## Passo 3 — Montar a amostra do gabarito (~120 discursos)
Amostra **estratificada** por tema previsto (cobre temas variados). Exporta
`gabarito_amostra.csv` com um índice e o texto, para rotularmos com a LLM no próximo passo.

In [ ]:
np.random.seed(42)
escolhidos = []
for j in range(32):                       # alguns discursos de cada tema previsto
    cand = np.where(pred[:, j] == 1)[0]
    if len(cand):
        escolhidos += list(np.random.choice(cand, size=min(4, len(cand)), replace=False))
escolhidos = list(dict.fromkeys(escolhidos))[:120]   # remove repetidos, limita a 120

amostra = disc.iloc[escolhidos].reset_index(drop=True)
amostra.insert(0, "idx", range(len(amostra)))
amostra[["idx", "nome_deputado", "transcricao"]].to_csv(
    "gabarito_amostra.csv", sep=";", index=False, encoding="utf-8")
# guarda os indices originais para casar com o BERTimbau depois
np.save("gabarito_idx.npy", np.array(escolhidos))
print(f"Amostra de {len(amostra)} discursos salva em gabarito_amostra.csv")

## Passo 4 — Rotular a amostra com a LLM (rota manual)

A célula abaixo **gera os prompts** em lotes (12 discursos por vez). Para cada lote:
1. Copie o texto impresso e **cole no Claude (claude.ai) ou ChatGPT**.
2. A LLM responde um **JSON** no formato `{"0": ["Saúde", "Educação"], "1": [...], ...}`.
3. Cole **todas** as respostas juntas (um dicionário só) no arquivo `gabarito_llm.json`
   nesta pasta (ou na célula seguinte).

O prompt obriga a LLM a escolher **somente** da lista dos 32 temas (igual aos rótulos).

In [ ]:
import textwrap

LISTA = "\n".join(f"- {t}" for t in nomes_temas)
amostra = pd.read_csv("gabarito_amostra.csv", sep=";", dtype=str)

INSTRUCAO = (
    "Você é um classificador de temas de discursos parlamentares. Para CADA discurso abaixo, "
    "liste os temas que ele aborda, escolhendo SOMENTE desta lista de 32 temas (use o nome exato):\n"
    f"{LISTA}\n\n"
    "Responda APENAS um JSON: a chave é o numero do discurso (idx) e o valor é a lista de temas. "
    'Ex.: {"0": ["Saúde"], "1": ["Economia", "Trabalho e Emprego"]}\n\n"'
)

LOTE = 12
for ini in range(0, len(amostra), LOTE):
    bloco = amostra.iloc[ini:ini+LOTE]
    print("=" * 90)
    print(f"### LOTE {ini//LOTE + 1} — cole tudo isto na LLM ###\n")
    print(INSTRUCAO)
    for _, r in bloco.iterrows():
        txt = r["transcricao"][:1500]      # limita p/ caber bem no chat
        print(f"[idx {r['idx']}] {txt}\n")

Depois de colar **todas** as respostas da LLM num único JSON (arquivo `gabarito_llm.json`
nesta pasta), rode a célula abaixo para transformá-las em multirrótulo.

In [ ]:
import json

with open("gabarito_llm.json", encoding="utf-8") as f:
    respostas = json.load(f)          # {"0": ["Saúde", ...], ...}

idx_tema = {t: j for j, t in enumerate(nomes_temas)}
Y_gab = np.zeros((len(amostra), 32), dtype=int)
nao_reconhecidos = set()
for k, temas in respostas.items():
    i = int(k)
    for t in temas:
        if t in idx_tema:
            Y_gab[i, idx_tema[t]] = 1
        else:
            nao_reconhecidos.add(t)

if nao_reconhecidos:
    print("AVISO: temas fora da lista (ignorados):", nao_reconhecidos)
np.save("gabarito_Y.npy", Y_gab)
print(f"Gabarito montado: {Y_gab.shape[0]} discursos x 32 temas; {int(Y_gab.sum())} marcacoes.")

## Passo 5 — Conferência humana (~30) e validação da LLM
Mostramos 30 discursos com os temas que a LLM atribuiu, para você **conferir por amostragem**.
Se a maioria estiver coerente, aceitamos a LLM como gabarito. (Se quiser corrigir algum,
edite o `gabarito_llm.json` e rode de novo a célula anterior.)

In [ ]:
rng = np.random.default_rng(0)
for i in rng.choice(len(amostra), size=min(30, len(amostra)), replace=False):
    temas = [nomes_temas[j] for j in np.where(Y_gab[i] == 1)[0]]
    print(f"[idx {i}] {amostra.iloc[i]['nome_deputado']}")
    print("  trecho:", amostra.iloc[i]["transcricao"][:160].replace("\n", " "))
    print("  LLM ->", " | ".join(temas) if temas else "(nenhum)", "\n")

## Passo 6 — Medir o *domain shift* (BERTimbau × gabarito), por tema
Comparamos a previsão do BERTimbau nos **mesmos** discursos da amostra com o gabarito da LLM.
Saída: macro-F1 nos discursos e **F1 por tema** (`resultados_dominio.csv`). Comparado com a
régua das ementas (0,70), mostra **quanto** o modelo perde ao mudar de domínio — e **em quais
temas** ele ainda é confiável (os que vamos privilegiar no cruzamento do notebook 06).

In [8]:
# Passo 6 e AUTOSSUFICIENTE: le tudo do disco, nao precisa do Passo 2 em memoria.
import json, numpy as np, pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_recall_fscore_support

# ordem dos 32 temas (= treino) e mascara de suporte >=200 (mesma das ementas)
props = pd.read_csv("proposicoes_temas.csv", dtype=str)
Yprop = MultiLabelBinarizer().fit_transform(props["temas"].str.split("|"))
nomes_temas = list(MultiLabelBinarizer().fit(props["temas"].str.split("|")).classes_)
idx_tema = {t: j for j, t in enumerate(nomes_temas)}
mask_sup200 = Yprop.sum(axis=0) >= 200

# GABARITO (LLM) -> multi-hot, na ordem idx 0..119
gab = json.load(open("gabarito_llm.json", encoding="utf-8"))
Y_gab = np.zeros((len(gab), 32), dtype=int)
for k, temas in gab.items():
    for t in temas:
        if t in idx_tema:
            Y_gab[int(k), idx_tema[t]] = 1

# PREVISAO do BERTimbau nos MESMOS discursos (gabarito_idx -> discursos_classificados.csv)
idx_orig = np.load("gabarito_idx.npy")
dc = pd.read_csv("discursos_classificados.csv", sep=";", dtype=str)
pred_amostra = np.zeros((len(idx_orig), 32), dtype=int)
for i, orig in enumerate(idx_orig):
    temas = str(dc.iloc[orig]["temas_previstos"])
    if temas and temas != "nan":
        for t in temas.split("|"):
            if t in idx_tema:
                pred_amostra[i, idx_tema[t]] = 1

# METRICAS
macro    = f1_score(Y_gab, pred_amostra, average="macro", zero_division=0)
macro200 = f1_score(Y_gab[:, mask_sup200], pred_amostra[:, mask_sup200], average="macro", zero_division=0)
micro    = f1_score(Y_gab, pred_amostra, average="micro", zero_division=0)
print("DOMAIN SHIFT (BERTimbau x gabarito, nos discursos da amostra):")
print(f"  macro-F1 (32)    = {macro:.3f}")
print(f"  macro-F1 (>=200) = {macro200:.3f}   <- ementas foram 0,700")
print(f"  micro-F1         = {micro:.3f}")
print(f"  temas/discurso -> BERTimbau: {pred_amostra.sum(1).mean():.2f} | gabarito: {Y_gab.sum(1).mean():.2f}")

p, r, f1v, sup = precision_recall_fscore_support(Y_gab, pred_amostra, zero_division=0)
dom = pd.DataFrame({"tema": nomes_temas, "f1_discurso": f1v.round(3),
                    "precisao": p.round(3), "revocacao": r.round(3),
                    "suporte_amostra": sup}).sort_values("f1_discurso", ascending=False)
dom.to_csv("resultados_dominio.csv", index=False, encoding="utf-8")
print("\nSalvo: resultados_dominio.csv (F1 por tema nos discursos)")
dom

DOMAIN SHIFT (BERTimbau x gabarito, nos discursos da amostra):
  macro-F1 (32)    = 0.525
  macro-F1 (>=200) = 0.569   <- ementas foram 0,700
  micro-F1         = 0.552
  temas/discurso -> BERTimbau: 5.92 | gabarito: 3.43

Salvo: resultados_dominio.csv (F1 por tema nos discursos)


,tema,f1_discurso,precisao,revocacao,suporte_amostra
12,Direito e Defesa do Consumidor,1.000,1.000,1.000,5
27,Relações Internacionais e Comércio Exterior,0.815,0.688,1.000,11
15,Economia,0.721,0.689,0.756,41
31,"Viação, Transporte e Mobilidade",0.720,0.600,0.900,10
30,Turismo,0.714,0.556,1.000,5
23,Meio Ambiente e Desenvolvimento Sustentável,0.710,0.647,0.786,14
11,Direito Penal e Processual Penal,0.704,0.576,0.905,21
7,Comunicações,0.700,0.583,0.875,8
16,Educação,0.667,0.625,0.714,14
29,Trabalho e Emprego,0.652,0.682,0.625,24


---
**Pronto (parte 1).** Geramos `discursos_classificados.csv` (todos os discursos rotulados) e
`resultados_dominio.csv` (confiabilidade por tema). O notebook **06** usa esses dois para o
cruzamento "fala vs. faz" e por partido/UF, privilegiando os temas confiáveis.